# OSACT5 Dataset: Download, Inspection, Cleaning & Validation

**A reproducible preprocessing pipeline for Arabic offensive-language / hate-speech research (EMNLP-quality)**

This notebook downloads the OSACT2022 (OSACT5) shared-task dataset, performs a full
data-quality audit, implements a configurable Arabic text-normalization pipeline, and
exports clean, analysis-ready CSV files together with a research-grade summary report.

**Structure**

1. Environment setup
2. Download raw data
3. Load data (auto delimiter detection)
4. Data inspection (missingness, duplicates, lengths, script statistics, plots)
5. Dataset validation (encoding, IDs, label consistency)
6. Arabic preprocessing pipeline (fully configurable)
7. Cleaning report (before/after, change counts)
8. Save cleaned datasets
9. Final research report

> Every step is deterministic and seeded. Every preprocessing operation is toggled by an
> explicit configuration flag so the pipeline is auditable and citable in a paper's
> "Data Preprocessing" section.


## Part 1 — Environment Setup

In [ ]:
!pip -q install emoji==2.12.1 ftfy==6.2.0 chardet==5.2.0 tqdm==4.66.4

In [ ]:
import os
import re
import sys
import csv
import json
import random
import hashlib
import unicodedata
import platform
import subprocess
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import emoji
import ftfy
import chardet
from tqdm.auto import tqdm

tqdm.pandas()


In [ ]:
SEED = 42

def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_global_seed(SEED)
print(f"Global seed fixed to {SEED}")


In [ ]:
def print_environment_report() -> None:
    '''Print a reproducibility-grade environment report (paper appendix material).'''
    print("=" * 60)
    print("ENVIRONMENT REPORT")
    print("=" * 60)
    print(f"Python        : {platform.python_version()}")
    print(f"Platform      : {platform.platform()}")
    print(f"pandas        : {pd.__version__}")
    print(f"numpy         : {np.__version__}")
    print(f"matplotlib    : {plt.matplotlib.__version__}")
    print(f"emoji         : {emoji.__version__}")
    print(f"ftfy          : {ftfy.__version__}")
    print(f"chardet       : {chardet.__version__}")
    print("=" * 60)

print_environment_report()


## Part 2 — Download the OSACT5 Dataset

Download the three official OSACT2022 shared-task files with `wget`, save them under
standardized local filenames, and verify that every download actually succeeded
(non-zero size, correct HTTP status, file present on disk)

In [ ]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

DATASET_URLS = {
    "train": "https://alt.qcri.org/resources/OSACT2022/OSACT2022-sharedTask-train.txt",
    "dev":   "https://alt.qcri.org/resources/OSACT2022/OSACT2022-sharedTask-dev.txt",
    "test":  "https://alt.qcri.org/resources/OSACT2022/OSACT2022-sharedTask-test-tweets.txt",
}

LOCAL_PATHS = {
    "train": os.path.join(DATA_DIR, "train.txt"),
    "dev":   os.path.join(DATA_DIR, "dev.txt"),
    "test":  os.path.join(DATA_DIR, "test.txt"),
}


In [ ]:
def download_file(url: str, out_path: str, force: bool = False) -> bool:
    '''Download `url` to `out_path` with wget and return True on verified success.'''
    if os.path.exists(out_path) and not force:
        print(f"[skip] {out_path} already exists ({os.path.getsize(out_path)} bytes)")
        return True

    cmd = ["wget", "-q", "--show-progress", "-O", out_path, url]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"[FAIL] wget exited with code {result.returncode} for {url}")
        print(result.stderr[-500:])
        return False

    return verify_download(out_path)


def verify_download(path: str) -> bool:
    '''A download is considered successful iff the file exists and is non-empty.'''
    if not os.path.exists(path):
        print(f"[FAIL] {path} was not created")
        return False
    size = os.path.getsize(path)
    if size == 0:
        print(f"[FAIL] {path} is empty (0 bytes)")
        return False
    print(f"[OK]   {path} downloaded successfully ({size:,} bytes)")
    return True


download_status = {}
for split, url in DATASET_URLS.items():
    print(f"\nDownloading '{split}' split...")
    download_status[split] = download_file(url, LOCAL_PATHS[split])

print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)
for split, ok in download_status.items():
    print(f"{split:6s}: {'SUCCESS' if ok else 'FAILED'}")

if not all(download_status.values()):
    raise RuntimeError(
        "One or more downloads failed. If you are running outside a network with access "
        "to alt.qcri.org, manually place train.txt / dev.txt / test.txt inside the "
        f"'{DATA_DIR}/' directory and re-run this cell (force=False will detect them)."
    )


## Part 3 — Load the Data

OSACT-style release files are typically tab-separated, but we do not hard-code this
assumption: we auto-detect the delimiter from the raw file using `csv.Sniffer`, with a
robust fallback heuristic (tab vs. comma frequency) if sniffing fails on messy tweet text.


In [ ]:
def detect_delimiter(path: str, sample_lines: int = 20) -> str:
    '''Auto-detect the field delimiter of a raw text file.

    Strategy:
      1. Try csv.Sniffer on a sample of lines.
      2. Fall back to comparing tab vs comma counts per line (tweets often contain
         literal commas in text, so tab is preferred when both are plausible).

    Robust to files shorter than `sample_lines` (uses itertools.islice instead of a
    fixed-size list comprehension with next(), which used to raise StopIteration on
    small files such as dev/test splits).
    '''
    from itertools import islice
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        sample = "".join(islice(f, sample_lines))

    if not sample.strip():
        raise ValueError(f"{path} appears to be empty — cannot detect delimiter")

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters="\t,;|")
        return dialect.delimiter
    except csv.Error:
        pass

    tab_count = sample.count("\t")
    comma_count = sample.count(",")
    if tab_count >= comma_count:
        return "\t"
    return ","


def load_split(path: str) -> pd.DataFrame:
    '''Load a raw OSACT split with auto-detected delimiter and robust parsing.'''
    delimiter = detect_delimiter(path)
    print(f"[{os.path.basename(path)}] detected delimiter: {repr(delimiter)}")

    df = pd.read_csv(
        path,
        sep=delimiter,
        engine="python",
        quoting=csv.QUOTE_NONE,
        encoding="utf-8",
        on_bad_lines="warn",
        dtype=str,          # keep everything as string first; we cast later where needed
        keep_default_na=False,
    )
    # Normalize column names (strip whitespace, lowercase-safe copy kept separately)
    df.columns = [c.strip() for c in df.columns]
    return df


raw = {}
for split, path in LOCAL_PATHS.items():
    raw[split] = load_split(path)

print("\nLoaded splits:", {k: v.shape for k, v in raw.items()})


In [ ]:
def describe_split(name: str, df: pd.DataFrame) -> None:
    print("=" * 60)
    print(f"SPLIT: {name}")
    print("=" * 60)
    print(f"Shape       : {df.shape}")
    print(f"Columns     : {list(df.columns)}")
    print("\nDtypes:")
    print(df.dtypes)
    print("\nFirst rows:")
    display(df.head())
    print()

for split, df in raw.items():
    describe_split(split, df)


## Part 4 — Data Inspection

We generate a full data-quality report per split: missing values, duplicate rows/texts/IDs,
empty texts, text-length statistics, Arabic vs. Latin character ratios, and counts of
emojis, URLs, hashtags, and mentions. We then plot label distributions and text-length
histograms.

We try to auto-detect the text column and (if present) an ID column and label column(s),
since exact OSACT column naming can vary slightly between releases.


In [ ]:
ID_CANDIDATES = ["tweet_id", "id", "twitter_id"]
TEXT_CANDIDATES = ["text", "tweet", "tweet_text", "content"]
LABEL_CANDIDATES = ["off_label", "hs_label", "offensive_label", "label", "off", "hs",
                     "hate_speech_label", "class"]

def find_column(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    return None

def resolve_schema(df: pd.DataFrame) -> dict:
    id_col = find_column(df.columns, ID_CANDIDATES)
    text_col = find_column(df.columns, TEXT_CANDIDATES)
    label_cols = [c for c in df.columns if c.lower() in LABEL_CANDIDATES]

    if text_col is None:
        text_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
    if id_col is None and len(df.columns) > 1:
        id_col = df.columns[0]

    return {"id_col": id_col, "text_col": text_col, "label_cols": label_cols}

schemas = {split: resolve_schema(df) for split, df in raw.items()}
for split, schema in schemas.items():
    print(f"{split:6s} -> {schema}")


In [ ]:
# Regex patterns for surface-feature counting
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\S+")
ARABIC_CHAR_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")
LATIN_CHAR_RE = re.compile(r"[A-Za-z]")

def count_emojis(text: str) -> int:
    return len([ch for ch in text if ch in emoji.EMOJI_DATA])

def inspect_split(name: str, df: pd.DataFrame, schema: dict) -> dict:
    text_col = schema["text_col"]
    id_col = schema["id_col"]
    texts = df[text_col].astype(str)

    lengths_chars = texts.str.len()
    lengths_words = texts.str.split().apply(len)

    report = {
        "split": name,
        "n_rows": len(df),
        "missing_values_total": int(df.isna().sum().sum()) + int((df == "").sum().sum()),
        "missing_values_by_column": {c: int((df[c].isna() | (df[c] == "")).sum()) for c in df.columns},
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_texts": int(texts.duplicated().sum()),
        "duplicate_ids": int(df[id_col].duplicated().sum()) if id_col else None,
        "empty_texts": int((texts.str.strip() == "").sum()),
        "text_length_chars": {
            "mean": float(lengths_chars.mean()),
            "median": float(lengths_chars.median()),
            "std": float(lengths_chars.std()),
            "min": int(lengths_chars.min()),
            "max": int(lengths_chars.max()),
        },
        "text_length_words": {
            "mean": float(lengths_words.mean()),
            "median": float(lengths_words.median()),
            "std": float(lengths_words.std()),
        },
        "arabic_char_ratio": float(texts.apply(lambda t: len(ARABIC_CHAR_RE.findall(t)) / max(len(t), 1)).mean()),
        "latin_char_ratio": float(texts.apply(lambda t: len(LATIN_CHAR_RE.findall(t)) / max(len(t), 1)).mean()),
        "total_emojis": int(texts.apply(count_emojis).sum()),
        "total_urls": int(texts.apply(lambda t: len(URL_RE.findall(t))).sum()),
        "total_hashtags": int(texts.apply(lambda t: len(HASHTAG_RE.findall(t))).sum()),
        "total_mentions": int(texts.apply(lambda t: len(MENTION_RE.findall(t))).sum()),
    }
    return report

inspection_reports = {}
for split, df in raw.items():
    inspection_reports[split] = inspect_split(split, df, schemas[split])
    print("=" * 60)
    print(f"INSPECTION REPORT: {split}")
    print("=" * 60)
    print(json.dumps(inspection_reports[split], indent=2, ensure_ascii=False))
    print()


In [ ]:
# Plot: label distributions
def plot_label_distributions(df: pd.DataFrame, schema: dict, split_name: str) -> None:
    label_cols = schema["label_cols"]
    if not label_cols:
        print(f"[{split_name}] no label columns detected — skipping label plot "
              f"(expected for the unlabeled test split).")
        return

    n = len(label_cols)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, col in zip(axes, label_cols):
        counts = df[col].value_counts()
        ax.bar(counts.index.astype(str), counts.values, color="#3B6E8F")
        ax.set_title(f"{split_name}: {col}")
        ax.set_xlabel("Class")
        ax.set_ylabel("Count")
        for tick in ax.get_xticklabels():
            tick.set_rotation(30)

    plt.tight_layout()
    plt.show()

for split, df in raw.items():
    plot_label_distributions(df, schemas[split], split)


In [ ]:
# Plot: text length histograms
def plot_length_histogram(df: pd.DataFrame, schema: dict, split_name: str) -> None:
    text_col = schema["text_col"]
    lengths = df[text_col].astype(str).str.len()

    plt.figure(figsize=(7, 4))
    plt.hist(lengths, bins=40, color="#8F3B5C", edgecolor="white")
    plt.title(f"{split_name}: text length distribution (characters)")
    plt.xlabel("Characters per tweet")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

for split, df in raw.items():
    plot_length_histogram(df, schemas[split], split)


## Part 5 — Dataset Validation

We now formally validate each split and print an itemized anomaly report. Checks:

- No structurally corrupted rows (wrong field count vs. header)
- All text is valid UTF-8
- ID column is unique (where present)
- Label values are within the expected/consistent set
- Class distribution is reported for sanity-checking imbalance


In [ ]:
def is_valid_utf8(s: str) -> bool:
    try:
        s.encode("utf-8").decode("utf-8")
        return True
    except (UnicodeEncodeError, UnicodeDecodeError):
        return False


def validate_split(name: str, path: str, df: pd.DataFrame, schema: dict) -> dict:
    anomalies = []

    # 1. Corrupted rows: re-scan raw file for lines whose field count != header field count
    delimiter = detect_delimiter(path)
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    header_fields = len(lines[0].split(delimiter))
    corrupted_lines = [
        i for i, line in enumerate(lines[1:], start=2)
        if len(line.rstrip("\n").split(delimiter)) != header_fields
    ]
    if corrupted_lines:
        anomalies.append(f"{len(corrupted_lines)} structurally corrupted line(s), "
                          f"e.g. lines {corrupted_lines[:5]}")

    # 2. UTF-8 validity
    text_col = schema["text_col"]
    invalid_utf8 = (~df[text_col].astype(str).apply(is_valid_utf8)).sum()
    if invalid_utf8:
        anomalies.append(f"{invalid_utf8} row(s) with invalid UTF-8 text")

    # 3. Unique IDs
    id_col = schema["id_col"]
    if id_col:
        dup_ids = df[id_col].duplicated().sum()
        if dup_ids:
            anomalies.append(f"{dup_ids} duplicated ID(s) in column '{id_col}'")
    else:
        anomalies.append("no ID column detected — uniqueness check skipped")

    # 4. Label consistency
    label_report = {}
    for col in schema["label_cols"]:
        values = df[col].value_counts(dropna=False)
        label_report[col] = values.to_dict()
        # Flag stray/rare label values (possible typos), heuristic: any value with <0.1% share
        rare = values[values < max(1, int(0.001 * len(df)))]
        if len(rare) > 0 and len(values) > 2:
            anomalies.append(f"column '{col}' has suspicious rare label values: {rare.to_dict()}")

    result = {
        "split": name,
        "n_corrupted_rows": len(corrupted_lines),
        "n_invalid_utf8": int(invalid_utf8),
        "n_duplicate_ids": int(df[id_col].duplicated().sum()) if id_col else None,
        "label_distribution": label_report,
        "anomalies": anomalies if anomalies else ["none detected"],
    }
    return result


validation_reports = {}
for split, df in raw.items():
    validation_reports[split] = validate_split(split, LOCAL_PATHS[split], df, schemas[split])
    print("=" * 60)
    print(f"VALIDATION REPORT: {split}")
    print("=" * 60)
    print(json.dumps(validation_reports[split], indent=2, ensure_ascii=False))
    print()


## Part 6 — Arabic Preprocessing Pipeline

A fully **configurable**, function-based pipeline. Every transformation is independently
toggleable via `PreprocessConfig`, so the exact preprocessing recipe used in the paper can
be stated unambiguously (and ablated).

**Design choices worth documenting in the paper:**

- We normalize Unicode (NFKC) to collapse compatibility variants (e.g., presentation-form
  Arabic letters) into canonical codepoints — this reduces spurious vocabulary sparsity
  without altering meaning.
- We normalize Alef variants (أ إ آ ٱ → ا) and Yeh variants (ى ئ → ي) because these are
  extremely common *orthographic* (not semantic) variations in social-media Arabic, and
  normalizing them is standard practice in Arabic NLP (e.g., Farasa, CAMeL Tools) — it
  reduces sparsity without changing meaning.
- **Teh Marbuta normalization (ة → ه) is OFF by default** and only justified if a
  downstream task is robust to gender/word-final ambiguity (ة often marks feminine nouns
  and merging it with ه can lose morphological signal). We expose it as an explicit flag
  and document the trade-off rather than silently applying it.
- We deliberately **do not** remove hashtags, emojis, punctuation, or numbers, since these
  frequently carry sentiment/offensive-language signal (e.g., emoji-based insults,
  hashtag topics, exclamation marks as intensifiers) that is directly relevant to hate
  speech / offensive language detection.


In [ ]:
from dataclasses import dataclass

@dataclass
class PreprocessConfig:
    '''Toggle each Arabic preprocessing step independently.'''
    unicode_normalize: bool = True        # NFKC normalization
    fix_encoding: bool = True             # ftfy mojibake repair
    remove_tatweel: bool = True           # remove Arabic kashida (ـ)
    remove_diacritics: bool = True        # remove tashkeel (َ ً ُ ٌ ِ ٍ ّ ْ)
    normalize_alef: bool = True           # أ إ آ ٱ -> ا
    normalize_yeh: bool = True            # ى ئ -> ي
    normalize_teh_marbuta: bool = False   # ة -> ه  (OFF by default, see markdown above)
    remove_urls: bool = True
    remove_mentions: bool = True
    remove_html: bool = True
    remove_extra_whitespace: bool = True

    # explicitly preserved (documented for clarity, not actually toggled off):
    # hashtags, emojis, punctuation, numbers


In [ ]:
#  Individual, composable preprocessing functions 

TATWEEL_RE = re.compile(r"\u0640")
DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")
ALEF_VARIANTS_RE = re.compile(r"[\u0622\u0623\u0625\u0671]")   # آ أ إ ٱ
YEH_VARIANTS_RE = re.compile(r"[\u0649\u0626]")                # ى ئ
TEH_MARBUTA_RE = re.compile(r"\u0629")                          # ة
HTML_TAG_RE = re.compile(r"<[^>]+>")
HTML_ENTITY_RE = re.compile(r"&[a-zA-Z#0-9]+;")
WHITESPACE_RE = re.compile(r"\s+")


def normalize_unicode(text: str) -> str:
    '''NFKC-normalize to collapse Unicode compatibility variants.'''
    return unicodedata.normalize("NFKC", text)


def fix_encoding(text: str) -> str:
    '''Repair mojibake / encoding artifacts (e.g. double-encoded UTF-8).'''
    return ftfy.fix_text(text)


def remove_tatweel(text: str) -> str:
    '''Remove the Arabic elongation character (kashida), purely stylistic.'''
    return TATWEEL_RE.sub("", text)


def remove_diacritics(text: str) -> str:
    '''Remove tashkeel (short vowel marks) — rarely used in social media text anyway.'''
    return DIACRITICS_RE.sub("", text)


def normalize_alef(text: str) -> str:
    '''Collapse Alef variants (أ إ آ ٱ) to bare Alef (ا). Orthographic, not semantic.'''
    return ALEF_VARIANTS_RE.sub("\u0627", text)


def normalize_yeh(text: str) -> str:
    '''Collapse Yeh variants (ى ئ) to standard Yeh (ي). Orthographic, not semantic.'''
    return YEH_VARIANTS_RE.sub("\u064A", text)


def normalize_teh_marbuta(text: str) -> str:
    '''Collapse Teh Marbuta (ة) to Heh (ه).

    CAUTION: this can remove a morphological (often gender) signal. Only enable this if
    your downstream task/tokenizer is known to benefit (e.g. matches a stemmed lexicon).
    '''
    return TEH_MARBUTA_RE.sub("\u0647", text)


def remove_urls(text: str) -> str:
    return URL_RE.sub(" ", text)


def remove_mentions(text: str) -> str:
    return MENTION_RE.sub(" ", text)


def remove_html(text: str) -> str:
    text = HTML_TAG_RE.sub(" ", text)
    text = HTML_ENTITY_RE.sub(" ", text)
    return text


def remove_extra_whitespace(text: str) -> str:
    return WHITESPACE_RE.sub(" ", text).strip()


In [ ]:
#  Master pipeline: applies enabled steps in a fixed, documented order 

def clean_text(text: str, config: PreprocessConfig) -> str:
    '''Apply the configured Arabic preprocessing pipeline to a single string.

    Order matters: encoding/unicode fixes first, structural removals (URLs/mentions/HTML)
    next, then character-level normalization, then final whitespace cleanup.
    Hashtags, emojis, punctuation, and numbers are never touched.
    '''
    if not isinstance(text, str):
        return text

    if config.fix_encoding:
        text = fix_encoding(text)
    if config.unicode_normalize:
        text = normalize_unicode(text)
    if config.remove_html:
        text = remove_html(text)
    if config.remove_urls:
        text = remove_urls(text)
    if config.remove_mentions:
        text = remove_mentions(text)
    if config.remove_tatweel:
        text = remove_tatweel(text)
    if config.remove_diacritics:
        text = remove_diacritics(text)
    if config.normalize_alef:
        text = normalize_alef(text)
    if config.normalize_yeh:
        text = normalize_yeh(text)
    if config.normalize_teh_marbuta:
        text = normalize_teh_marbuta(text)
    if config.remove_extra_whitespace:
        text = remove_extra_whitespace(text)

    return text


def clean_dataframe(df: pd.DataFrame, text_col: str, config: PreprocessConfig) -> pd.DataFrame:
    '''Return a copy of df with an added 'text_clean' column.'''
    out = df.copy()
    out["text_clean"] = out[text_col].astype(str).progress_apply(lambda t: clean_text(t, config))
    return out


# Instantiate the default configuration used for this paper's experiments
default_config = PreprocessConfig()
print(default_config)


## Part 7 — Cleaning Report

We apply the pipeline to every split, show concrete before/after examples, count how many
rows were actually altered by cleaning, and summarize the effect of each individual step.


In [ ]:
cleaned = {}
for split, df in raw.items():
    text_col = schemas[split]["text_col"]
    cleaned[split] = clean_dataframe(df, text_col, default_config)
    n_changed = (cleaned[split]["text_clean"] != cleaned[split][text_col]).sum()
    print(f"[{split}] rows changed by cleaning: {n_changed} / {len(cleaned[split])} "
          f"({n_changed / len(cleaned[split]):.1%})")


In [ ]:
#  Before / after qualitative examples 
def show_before_after(df: pd.DataFrame, text_col: str, n: int = 5, seed: int = SEED) -> None:
    changed = df[df["text_clean"] != df[text_col]]
    sample = changed.sample(min(n, len(changed)), random_state=seed) if len(changed) else changed
    for i, row in sample.iterrows():
        print("-" * 60)
        print("BEFORE:", row[text_col])
        print("AFTER :", row["text_clean"])

print("Example transformations (train split):")
show_before_after(cleaned["train"], schemas["train"]["text_col"])


In [ ]:
#  Per-step impact summary table 
def summarize_step_impact(df: pd.DataFrame, text_col: str, config: PreprocessConfig) -> pd.DataFrame:
    '''For each enabled step, count how many rows it individually alters (single-step ablation).'''
    base_texts = df[text_col].astype(str)
    steps = {
        "fix_encoding": fix_encoding,
        "unicode_normalize": normalize_unicode,
        "remove_html": remove_html,
        "remove_urls": remove_urls,
        "remove_mentions": remove_mentions,
        "remove_tatweel": remove_tatweel,
        "remove_diacritics": remove_diacritics,
        "normalize_alef": normalize_alef,
        "normalize_yeh": normalize_yeh,
        "normalize_teh_marbuta": normalize_teh_marbuta,
        "remove_extra_whitespace": remove_extra_whitespace,
    }

    rows = []
    for step_name, fn in steps.items():
        enabled = getattr(config, step_name)
        if not enabled:
            rows.append({"step": step_name, "enabled": enabled, "rows_affected": None})
            continue
        transformed = base_texts.apply(fn)
        n_affected = int((transformed != base_texts).sum())
        rows.append({"step": step_name, "enabled": enabled, "rows_affected": n_affected})

    return pd.DataFrame(rows)

summary_table = summarize_step_impact(cleaned["train"], schemas["train"]["text_col"], default_config)
print("Per-step impact on TRAIN split (single-step ablation):")
display(summary_table)


## Part 8 — Save Cleaned Datasets

Export the cleaned splits as UTF-8 CSV files, ready for tokenization / model training.


In [ ]:
OUTPUT_DIR = "clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATHS = {
    "train": os.path.join(OUTPUT_DIR, "train_clean.csv"),
    "dev":   os.path.join(OUTPUT_DIR, "dev_clean.csv"),
    "test":  os.path.join(OUTPUT_DIR, "test_clean.csv"),
}

for split, df in cleaned.items():
    df.to_csv(OUTPUT_PATHS[split], index=False, encoding="utf-8")
    print(f"[saved] {OUTPUT_PATHS[split]}  ({len(df)} rows, {df.shape[1]} columns)")


## Part 9 — Final Research Report

A single consolidated summary suitable for the "Dataset" section of a paper.


In [ ]:
def build_research_report(raw: dict, cleaned: dict, schemas: dict,
                           inspection_reports: dict) -> pd.DataFrame:
    rows = []
    for split in raw:
        df_raw = raw[split]
        df_clean = cleaned[split]
        text_col = schemas[split]["text_col"]
        label_cols = schemas[split]["label_cols"]

        total_rows = len(df_raw)
        dup_rows_removed = int(df_raw.duplicated().sum())
        missing_values = inspection_reports[split]["missing_values_total"]

        lengths = df_clean["text_clean"].astype(str).str.len()
        avg_len = float(lengths.mean())
        median_len = float(lengths.median())

        vocab = set()
        for t in df_clean["text_clean"].astype(str):
            vocab.update(t.split())
        vocab_size = len(vocab)

        label_dist = {}
        for col in label_cols:
            label_dist[col] = df_raw[col].value_counts().to_dict()

        rows.append({
            "split": split,
            "total_rows": total_rows,
            "duplicate_rows_found": dup_rows_removed,
            "missing_values": missing_values,
            "avg_text_length_chars": round(avg_len, 2),
            "median_text_length_chars": median_len,
            "vocabulary_size": vocab_size,
            "label_distribution": json.dumps(label_dist, ensure_ascii=False),
        })

    return pd.DataFrame(rows)


research_report = build_research_report(raw, cleaned, schemas, inspection_reports)

print("=" * 70)
print("RESEARCH REPORT — OSACT5 DATASET SUMMARY")
print("=" * 70)
display(research_report)

research_report.to_csv(os.path.join(OUTPUT_DIR, "research_report.csv"), index=False, encoding="utf-8")
print(f"\n[saved] {os.path.join(OUTPUT_DIR, 'research_report.csv')}")
